# Data Quality Check — NYC Parking Violations

Dieses Notebook dokumentiert die Datenqualitätsprüfung, die während der Analyse entdeckten Probleme sowie die daraus abgeleiteten Massnahmen für das Preprocessing.

In [1]:
# Setup
from pyspark.sql import SparkSession
import pyspark.sql.functions as f
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

spark = SparkSession.builder \
    .appName("BDLC_DataQualityCheck") \
    .master("spark://bdlc-012.bdlc.ls.eee.intern:7077") \
    .config("spark.executor.cores", "2") \
    .config("spark.executor.memory", "8g") \
    .config("spark.cores.max", "18") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")
spark

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/05/23 17:44:14 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/05/23 17:44:15 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


26/05/23 17:44:25 WARN GarbageCollectionMetrics: To enable non-built-in garbage collector(s) List(G1 Concurrent GC), users should configure it(them) to spark.eventLog.gcMetrics.youngGenerationGarbageCollectors or spark.eventLog.gcMetrics.oldGenerationGarbageCollectors


In [ ]:
# Daten laden
processed_path = "hdfs:///parking_violations/processed/parking_violations_cleaned"
df = spark.read.parquet(processed_path)

total = df.count()
print(f"Gesamtanzahl Records: {total:,}")

## 1 — Jahresverteilung (Übersicht)

Erste Prüfung: Wie verteilen sich die Records auf Kalenderjahre? Gibt es offensichtlich fehlerhafte Jahreszahlen?

In [ ]:
# Alle Kalenderjahre mit Anzahl Records anzeigen
df.withColumn("issue_year_extracted", f.year("issue_date_parsed")) \
    .groupBy("issue_year_extracted") \
    .count() \
    .orderBy("issue_year_extracted") \
    .show(50)

**Befund:** Es existieren Records mit Jahreszahlen von 1972 bis 2066 — offensichtliche Tippfehler im Originaldatensatz. Die validen Jahre sind 2022–2025. Die Anzahl fehlerhafter Records ist jedoch minimal (< 1'000 von 54 Millionen).

## 2 — Spike-Analyse: FY2023 Juli–September

Bei der monatlichen Analyse nach Fiskaljahr wurde ein auffälliger Spike in FY2023 Monate 7–9 entdeckt. Hier die ursprüngliche (fehlerhafte) Darstellung:

In [ ]:
# Ursprüngliche Analyse nach Fiskaljahr (zeigt den Spike)
monthly_fy = df.groupBy("fiscal_year", "issue_month") \
    .count() \
    .orderBy("fiscal_year", "issue_month") \
    .toPandas()

month_labels = {1:"Jan", 2:"Feb", 3:"Mär", 4:"Apr", 5:"Mai", 6:"Jun",
                7:"Jul", 8:"Aug", 9:"Sep", 10:"Okt", 11:"Nov", 12:"Dez"}

pivot_fy = monthly_fy.pivot(index="issue_month", columns="fiscal_year", values="count")
pivot_fy.index = pivot_fy.index.map(month_labels)

fig, ax = plt.subplots(figsize=(12, 5))
pivot_fy.plot(marker="o", ax=ax)
ax.set_title("[FEHLERHAFT] Violations pro Monat nach Fiskaljahr — Spike sichtbar",
             fontsize=12, fontweight="bold", color="red")
ax.set_xlabel("Monat")
ax.set_ylabel("Anzahl Violations")
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x/1e6:.1f}M"))
ax.grid(True, alpha=0.4)
ax.legend(title="Fiskaljahr")
plt.tight_layout()
plt.savefig("dq_spike_fehlerhaft.png", dpi=150)
plt.show()

### Untersuchung des Spikes

Welche Kalenderjahre stecken hinter FY2023 Monate 7–9?

In [ ]:
# Aufschlüsselung nach Kalenderjahr für den Spike-Bereich
df.filter(
    (f.col("fiscal_year") == 2023) &
    (f.col("issue_month").isin([7, 8, 9]))
) \
.withColumn("calendar_year", f.year("issue_date_parsed")) \
.groupBy("calendar_year", "issue_month") \
.count() \
.orderBy("calendar_year", "issue_month") \
.filter(f.col("calendar_year") >= 2022) \
.show()

**Befund:** FY2023 Monate 7–9 enthält Records aus sowohl Kalenderjahr 2022 als auch 2023. Das deutet auf doppelte Records hin — die gleichen Violations erscheinen in mehreren Fiskaljahr-Files.

## 3 — Duplikat-Analyse

Prüfung ob `summons_number` (Primary Key) doppelt vorkommt:

In [ ]:
# Duplikat-Check über gesamten Datensatz
total = df.count()
unique = df.select("summons_number").distinct().count()
duplicates = total - unique

print(f"Total Records       : {total:,}")
print(f"Unique summons_number: {unique:,}")
print(f"Duplikate           : {duplicates:,}  ({duplicates/total*100:.1f}%)")

In [ ]:
# Bestätigung: Vorher/Nachher Vergleich für Juli-August 2023
df_dedup = df.dropDuplicates(["summons_number"])

print("=== VORHER (mit Duplikaten) ===")
df.withColumn("calendar_year", f.year("issue_date_parsed")) \
    .filter((f.col("calendar_year") == 2023) & (f.col("issue_month").isin([7, 8]))) \
    .groupBy(f.month("issue_date_parsed").alias("monat")) \
    .count().orderBy("monat").show()

print("=== NACHHER (ohne Duplikate) ===")
df_dedup.withColumn("calendar_year", f.year("issue_date_parsed")) \
    .filter((f.col("calendar_year") == 2023) & (f.col("issue_month").isin([7, 8]))) \
    .groupBy(f.month("issue_date_parsed").alias("monat")) \
    .count().orderBy("monat").show()

**Befund:** Die Counts halbieren sich nach der Deduplizierung exakt (Faktor ×2). Die Duplikate sind die alleinige Ursache des Spikes — jede Violation aus Juli–August 2023 erscheint genau zweimal im kombinierten Dataset (einmal in FY2023-File, einmal in FY2024-File).

## 4 — Massnahmen im Preprocessing

Folgende zwei Korrekturen wurden im Preprocessing-Notebook ergänzt:

**Massnahme 1 — Duplikate entfernen:**
```python
df = df.dropDuplicates(["summons_number"])
```

**Massnahme 2 — Fehlerhafte Datumswerte filtern:**
```python
df = df.filter(
    (f.col("issue_date_parsed") >= "2022-01-01") &
    (f.col("issue_date_parsed") <= "2025-12-31")
)
```

Reihenfolge: erst deduplizieren, dann filtern, dann Parquet neu speichern.

In [ ]:
spark.stop()